In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
from functools import partial
import json
import random
import torch
from datasets import Dataset, load_dataset, concatenate_datasets, interleave_datasets
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, TrainerCallback
from torch.utils.data import DataLoader
from tqdm import tqdm
from trl import SFTTrainer, SFTConfig
import wandb
import numpy as np

import sys
from pathlib import Path

PROJECT_ROOT = Path("../")
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")


/home/aladdin/miniconda3/envs/affine/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current working directory: /home/aladdin/Affine-H


In [4]:
from src.data import (
    get_abd_dataset,
    get_ded_dataset,
    get_mth_dataset,
    get_sci_dataset, 
    get_lgc_dataset,
    load_jsonl
)

from src.config import (
    AFFINE_DATASET_REPOSITORY,
    PRIME_DATASET_REPOSITORY,
    LGC_DATASET_REPOSITORY,
    AGENTGYM_DATASET_REPOSITORY,
    TRAIN_DATA_PATH,
)

agentgym_dataset = Dataset.from_list(load_jsonl(TRAIN_DATA_PATH + "/agentgym.jsonl")).remove_columns("task_id")

swe_dataset = Dataset.from_list(load_jsonl(TRAIN_DATA_PATH + "/swe.jsonl")).remove_columns("task_id")
cde_dataset = Dataset.from_list(load_jsonl(TRAIN_DATA_PATH + "/cde.jsonl")).remove_columns("task_id")

abd_dataset = get_abd_dataset(AFFINE_DATASET_REPOSITORY).remove_columns("task_id")
ded_dataset = get_ded_dataset(AFFINE_DATASET_REPOSITORY).remove_columns("task_id")

mth_dataset = get_mth_dataset(PRIME_DATASET_REPOSITORY).remove_columns("task_id")
sci_dataset = get_sci_dataset(PRIME_DATASET_REPOSITORY).remove_columns("task_id")
lgc_dataset = get_lgc_dataset(LGC_DATASET_REPOSITORY).remove_columns("task_id")


from dotenv import load_dotenv
load_dotenv()

True

In [5]:

train_dataset = lgc_dataset

train_dataset = train_dataset.rename_column(
    "conversations", "messages"
)

eval_dataset = train_dataset.shuffle(seed=42).select(range(min(50, len(train_dataset))))

print("="*60)
print(f"Train dataset size: {len(train_dataset)}")
print(f"AgentGym: {len(agentgym_dataset)}")
print(f"LGC: {len(lgc_dataset)}")
print(f"ABD: {len(abd_dataset)}")
print(f"DED: {len(ded_dataset)}")
print(f"MTH: {len(mth_dataset)}")
print(f"SCI: {len(sci_dataset)}")
print(f"SWE: {len(swe_dataset)}")
print(f"CDE: {len(cde_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")
print("="*60)


Train dataset size: 81566
AgentGym: 3018
LGC: 81566
ABD: 23303
DED: 23303
MTH: 21161
SCI: 29307
SWE: 419
CDE: 656
Eval dataset size: 50


In [6]:
# Import PEFT for LoRA
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = os.getenv("MODEL_NAME")
OUTPUT_DIR = "./sft_output_lora"

CHAT_TEMPLATE = r"""
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if message.content is string %}
        {%- set content = message.content %}
    {%- else %}
        {%- set content = '' %}
    {%- endif %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}
        {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {%- generation -%}
        {{- '<|im_start|>' + message.role + '\n' + content }}
        {%- if message.tool_calls %}
            {%- for tool_call in message.tool_calls %}
                {%- if (loop.first and content) or (not loop.first) %}
                    {{- '\n' }}
                {%- endif %}
                {%- if tool_call.function %}
                    {%- set tool_call = tool_call.function %}
                {%- endif %}
                {{- '<tool_call>\n{"name": "' }}
                {{- tool_call.name }}
                {{- '", "arguments": ' }}
                {%- if tool_call.arguments is string %}
                    {{- tool_call.arguments }}
                {%- else %}
                    {{- tool_call.arguments | tojson }}
                {%- endif %}
                {{- '}\n</tool_call>' }}
            {%- endfor %}
        {%- endif %}
        {{- '<|im_end|>\n' }}
        {%- endgeneration -%}
    {%- elif message.role == "tool" %}
        {%- if loop.first or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
""".strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.chat_template = CHAT_TEMPLATE

# Load model with LoRA-friendly settings
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=False,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.config.use_cache = False

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                           # LoRA rank (you can adjust: 8, 16, 32, 64)
    lora_alpha=32,                  # LoRA alpha (typically 2*r)
    lora_dropout=0.1,               # Dropout probability
    target_modules=[                # Target modules to apply LoRA
        "q_proj",
        "k_proj", 
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    inference_mode=False,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# If we added a token, resize embeddings
if model.config.vocab_size != len(tokenizer):
    model.resize_token_embeddings(len(tokenizer))



`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.30s/it]


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [7]:
from src.config import get_sampling_range

sampling_range = get_sampling_range()

env_keymap = {
    "MTH": "mth",
    "SCI": "sci",
    "LGC": "lgc",
    "CDE": "cde",
    "agentgym:webshop": "webshop",
    "agentgym:alfworld": "alfworld",
    "agentgym:sciworld": "sciworld",
    "agentgym:textcraft": "textcraft",  
    "affine:abd-v2": "abd-v2",
    "affine:ded-v2": "ded-v2",
    "SWE-PRO": "swe-pro",
}

eval_ids = {env_keymap[k]: random.sample(v['sampling_config']['sampling_list'], k=1) for k, v in sampling_range.items()}

In [8]:
eval_ids.pop('swe-pro', None)

[29]

In [14]:
from src.evaluation import evaluate_env

👋 This is mini-swe-agent version 1.17.3.
Loading global config from '/home/aladdin/.config/mini-swe-agent/.env'

In [ ]:
tokenizer.decode([151645])

In [ ]:
from src.evaluation import evaluate_env
from transformers import TrainerCallback

class MetricsCallback(TrainerCallback):
    def __init__(self, model, tokenizer, eval_ids, eval_steps=8):
        super().__init__()
        self.model = model
        self.tokenizer = tokenizer
        self.eval_ids = eval_ids
        self.eval_steps = eval_steps
        
    async def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_steps == 0:
            model.eval()
            wandb.log({"step": state.global_step})
            scores = []
            with torch.no_grad():
                for env, task_ids in eval_ids.items():
                    result = await evaluate_env(model, env=env, task_ids=task_ids, tokenizer=tokenizer)
                    score = result.get("score", 0.0)
                    wandb.log({f"{env}_score": score}, step=state.global_step)
                    scores.append(score)
            wandb.log({"mean_score": sum(scores) / len(scores)}, step=state.global_step)
            model.train()
            

In [9]:
import torch
import wandb
from transformers import TrainerCallback

class MetricsCallback(TrainerCallback):
    def __init__(self, eval_steps, model, tokenizer, eval_ids):
        super().__init__()
        self.eval_steps = eval_steps
        self.model = model
        self.tokenizer = tokenizer
        self.eval_ids = eval_ids

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 0:
            return control
        
        if state.global_step % self.eval_steps == 0:
            # Run evaluation synchronously
            self._run_eval(state.global_step)
        
        return control
    
    def _run_eval(self, step):
        """Synchronous wrapper that runs async evaluation"""
        import asyncio
        asyncio.run(self._async_eval(step))
    
    async def _async_eval(self, step):
        self.model.eval()
        scores = []
        with torch.no_grad():
            for env, task_ids in self.eval_ids.items():
                result = await evaluate_env(
                    self.model,
                    env=env,
                    task_ids=task_ids,
                    tokenizer=self.tokenizer,
                )
                score = result.get("score", 0.0)
                wandb.log({f"{env}_score": score}, step=step)
                scores.append(score)
        
        if scores:
            wandb.log({"mean_score": sum(scores) / len(scores)}, step=step)
        
        self.model.train()

In [ ]:
with torch.no_grad():
    for env, task_ids in eval_ids.items():
        score = await evaluate_env(model, env=env, task_ids=task_ids, tokenizer=tokenizer)
        print(f"{env}: {score['score']}")

In [ ]:
tokenizer.chat_template = CHAT_TEMPLATE

In [10]:
eval_dataset = train_dataset.shuffle(seed=42).select(range(min(16, len(train_dataset))))
train_dataset = train_dataset.shuffle(seed=42).select(range(min(128, len(train_dataset))))


In [12]:

NUM_PROC = 8

training_args = SFTConfig(
    model_init_kwargs={"torch_dtype": torch.bfloat16},
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=15,
    learning_rate=1e-05,
    warmup_steps=32,
    max_length=8192,
    max_grad_norm=1.0,
    dataset_num_proc=8,
    
    # Conversation control
    assistant_only_loss=True,
            
    # Logging
    logging_strategy="steps",
    logging_steps=8,
    
    # Evaluation
    do_eval=True,
    eval_strategy="steps",
    eval_steps=8,
    save_total_limit=3,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Checkpointing
    output_dir=OUTPUT_DIR,
    save_strategy="steps",
    save_steps=8,
    bf16=True,
    remove_unused_columns=False,
    report_to=["wandb"],
    
    # Memory optimization
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)


# Initialize wandb
wandb.init(
    project="SFT - lora",
    name="LGC improvement",
    config={
        "model_name": MODEL_NAME,
        "train_dataset": "LGC",
        "lora_r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "training_method": "LoRA",
        **training_args.to_dict(),
    }
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[MetricsCallback(eval_steps=8, model=model, tokenizer=tokenizer, eval_ids=eval_ids)],
)

wandb: Currently logged in as: hongyuxiao05 (hongyuxiao05-lambda) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


You passed `model_init_kwargs` to the `SFTConfig`, but your model is already instantiated. The `model_init_kwargs` will be ignored.
Truncating eval dataset (num_proc=8): 100%|██████████| 16/16 [00:00<00:00, 25.24 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
/home/aladdin/miniconda3/envs/affine/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/aladdin/miniconda3/envs/affine/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [11]:
eval_ids = {env_keymap[k]: random.sample(v['sampling_config']['sampling_list'], k=5) for k, v in sampling_range.items()}

In [15]:

trainer.train()

# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR + "/final_lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR + "/final_lora_adapter")

wandb.finish()

print("Training complete! LoRA adapter saved to:", OUTPUT_DIR + "/final_lora_adapter")

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
8,1.033200,0.735298,0.148739,45242.000000,0.931450


2025-12-24 07:20:22 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 07:20:22 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 07:20:28 - i3_math - INFO - Dataset loaded: 21161 examples
2025-12-24 07:20:28 - i3_math - INFO - Dataset loaded: 21161 examples
2025-12-24 07:20:31 - i3_math - INFO - Evaluation complete: score=1.0, state={'math_verify_score': 1.0}
2025-12-24 07:20:31 - i3_math - INFO - Evaluation complete: score=1.0, state={'math_verify_score': 1.0}
2025-12-24 07:20:32 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 07:20:32 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 07:20:36 - i3_math - INFO - Dataset loaded: 21161 examples
2025-12-24 07:20:36 - i3_math - INFO - Dataset loaded: 21161 examples
2025-12-24 07:20:38 - i3_math - INFO - Evaluation complete: score=1.0, state={'math_verify_score': 1

response ["WebShop [SEP] Instruction: [SEP] Find me machine wash men's dress shirts with cotton spandex, classic fit, short sleeve with color: melon berry, and size: 3x, and price lower than 60.00 dollars [SEP] Search", None]
search[machine wash men's dress shirts cotton spandex classic fit short sleeve melon berry 3x price < 60.00]
click[B07HRFSNL4]
click[3x]
click[melon berry]
click[Buy Now]
response ["WebShop [SEP] Instruction: [SEP] Find me men's t-shirts & tanks with short sleeve, fashion design, long sleeve, button closure with color: b-gray, and size: 4x-large, and price lower than 30.00 dollars [SEP] Search", None]
search[men's t-shirts tanks short sleeve fashion design long sleeve button closure color:b-gray size:4x-large price < 30.00]
click[B078WSND96]
click[b-gray]
click[4x-large]
click[Buy Now]
response ["WebShop [SEP] Instruction: [SEP] Find me machine wash men's dress shirts with polyester heathers, heathers cotton, cotton heather, needle sleeve, classic fit with color: 

2025-12-24 07:40:12 - i3_code - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/code split=train
2025-12-24 07:40:18 - i3_code - INFO - Dataset loaded: 8579 examples
2025-12-24 07:44:24 - i3_code - INFO - Evaluation start: task_id=38, model=N/A, base_url=N/A
2025-12-24 07:44:36 - i3_code - INFO - Evaluation complete: task_id=38, model=N/A, base_url=N/A, 17/17, score=1.0
2025-12-24 07:44:36 - i3_code - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/code split=train
2025-12-24 07:44:42 - i3_code - INFO - Dataset loaded: 8579 examples
2025-12-24 07:47:45 - i3_code - INFO - Evaluation start: task_id=107, model=N/A, base_url=N/A
2025-12-24 07:47:50 - i3_code - INFO - Evaluation complete: task_id=107, model=N/A, base_url=N/A, 8/8, score=1.0
2025-12-24 07:47:51 - i3_code - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/code split=train
2025-12-24 07:47:55 - i3_code - INFO - Dataset loaded: 8579 examples
2025-12-24 08:06:44 - i3_code - INFO - Evaluation start: task_id=179, 

The text is not in the correct format. Parsing result may not be accurate.
###RAW TEXT:
 Action:
open door to kitchen

###PARSED THOUGHT:
 

###PARSED ACTION:
 open door to kitchen
The text is not in the correct format. Parsing result may not be accurate.
###RAW TEXT:
 Action:
go to kitchen

###PARSED THOUGHT:
 

###PARSED ACTION:
 go to kitchen
The text is not in the correct format. Parsing result may not be accurate.
###RAW TEXT:
 Action:
pick up thermometer

###PARSED THOUGHT:
 

###PARSED ACTION:
 pick up thermometer
The text is not in the correct format. Parsing result may not be accurate.
###RAW TEXT:
 Action:
focus on thermometer in inventory

###PARSED THOUGHT:
 

###PARSED ACTION:
 focus on thermometer in inventory
The text is not in the correct format. Parsing result may not be accurate.
###RAW TEXT:
 Action:
open door to bathroom

###PARSED THOUGHT:
 

###PARSED ACTION:
 open door to bathroom
The text is not in the correct format. Parsing result may not be accurate.
###RAW T

minisweagent.environment: DEBUG: Starting container with command: docker run -d --name minisweagent-32c8423e -w    
/app --rm --entrypoint '' --name swebench-pro-439-1766564543933                                                    
jefzda/sweap-images:internetarchive.openlibrary-internetarchive__openlibrary-3c48b4bb782189e0858e6c3fc7956046cf3e1c
fb-v2d9a6c849c60ed19fd0858ce9e40b sleep 1800

... (11 outputs left out)

/home/aladdin/miniconda3/envs/affine/lib/python3.11/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
2025-12-24 08:42:02 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 08:42:02 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 08:42:07 - i3_math - INFO - Dataset loaded: 21161 examples
2025-12-24 08:42:07 - i3_math - INFO - Dataset loaded: 21161 examples
2025-12-24 08:42:09 - i3_math - INFO - Evaluation complete: score=1.0, state={'math_verify_score': 1.0}
2025-12-24 08:42:09 - i3_math - INFO - Evaluation complete: score=1.0, state={'math_verify_score': 1.0}
2025-12-24 08:42:09 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24 08:42:09 - i3_math - INFO - Loading dataset: PrimeIntellect/INTELLECT-3-RL/math split=train
2025-12-24

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7114b64dbb90>> (for post_run_cell), with arguments args (<ExecutionResult object at 71159ef57390, execution_count=15 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 71159ef57490, raw_cell="
trainer.train()

# Save the LoRA adapter
model.sa.." transformed_cell="trainer.train()

# Save the LoRA adapter
model.sav.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu2204/home/aladdin/Affine-H/scripts/train.ipynb#X20sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
!pip install bitsandbytes